# AgentSynth — teach a base model function-calling (Colab)

Generate **verified** agent trajectories with AgentSynth, distill them into
function-calling supervision, and fine-tune a small **base** (non-instruct) model —
one that scores ~0% at function calling out of the box — then print the before/after
table on the built-in suite and a real BFCL slice. Runs on a **free Colab T4**.

Why a base model? Instruct models like Llama-3.2-1B-Instruct already score 90%+ on
the simple suites, so there's no headroom to demonstrate anything. A base model
can't emit a valid tool call at all — until it's trained on AgentSynth data.

First: **Runtime → Change runtime type → T4 GPU**.


## 1. Install


In [ ]:
%pip install -q "agentsynth-ai[train,hub]>=0.2.1" unsloth
# Bleeding edge? Install from main instead:
# %pip install -q "agentsynth-ai[train,hub] @ git+https://github.com/agentsynth/agentsynth" unsloth


## 2. Generate verified trajectories → function-calling supervision (CPU, no API key)

Each record asks the model to pick one tool for a query (the exact format the
benchmark uses) and supervises it with the trajectory's **first verified tool call**
— a call that actually ran in the environment. Mock generation by default; set a
provider key (e.g. `os.environ['ANTHROPIC_API_KEY']`) and drop the force-mock line
for richer real-LLM data.


In [ ]:
import os

os.environ.setdefault('AGENTSYNTH_FORCE_MOCK', '1')

import json
import random
from agentsynth import Recipe, run_recipe

result = run_recipe(Recipe(num_trajectories=500, vary_modes=True,
                           verify=True, dedup=True, rubric='strict'))
print('trajectories:', len(result.trajectories),
      '| pass@1:', result.metrics['pass_rate'],
      '| verified:', result.metrics.get('verified_rate'))

# The same prompt the benchmark's `prompted_model` uses — train and eval must
# speak one format, or the fine-tune scores 0 for the wrong reason.
PROMPT = ('You can call exactly one tool to help the user.\n'
          'Tools (JSON): {tools}\n\n'
          'User: {query}\n'
          'Respond with ONLY a JSON object: '
          '{{"tool": "<tool name>", "args": {{<arguments>}}}}')

rng = random.Random(7)
records, dpo_rows = [], []
for traj in result.trajectories:
    calls = traj.tool_calls()
    if not calls:
        continue
    tools = [{'name': t.name, 'description': t.description, 'parameters': t.parameters}
             for t in traj.tools]
    prompt = PROMPT.format(tools=json.dumps(tools), query=traj.query)
    gold = {'tool': calls[0].tool_name, 'args': calls[0].tool_args or {}}
    records.append({'prompt': prompt, 'completion': json.dumps(gold)})
    wrong = [t['name'] for t in tools if t['name'] != gold['tool']]
    if wrong:  # hard negative: right format/args, wrong tool
        dpo_rows.append({'prompt': prompt, 'chosen': json.dumps(gold),
                         'rejected': json.dumps({'tool': rng.choice(wrong),
                                                 'args': gold['args']})})

with open('tool_sft.jsonl', 'w') as fh:
    for r in records:
        fh.write(json.dumps(r) + '\n')
with open('dpo.jsonl', 'w') as fh:
    for r in dpo_rows:
        fh.write(json.dumps(r) + '\n')
print('tool-SFT records:', len(records), '| DPO pairs:', len(dpo_rows))


## 3. Load a small **base** model and benchmark it (before)

`Llama-3.2-1B` (no instruction tuning) — it has never been taught to emit a tool
call, so expect ~0% here. That's the headroom the fine-tune will fill.


In [ ]:
from unsloth import FastLanguageModel

BASE = 'unsloth/Llama-3.2-1B'  # base, NOT -Instruct: no function-calling skill yet
model, tokenizer = FastLanguageModel.from_pretrained(
    BASE, max_seq_length=2048, load_in_4bit=True)

def hf_complete(prompt):
    # Plain completion (a base model has no chat template).
    FastLanguageModel.for_inference(model)
    ids = tokenizer(prompt, return_tensors='pt').input_ids.to(model.device)
    out = model.generate(input_ids=ids, max_new_tokens=80, do_sample=False,
                         pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

from agentsynth.benchmarks import load_sample_bfcl, prompted_model, report_table_md, run_benchmark

bfcl = load_sample_bfcl()  # a real 25-case slice of the BFCL simple_python split
before = run_benchmark(prompted_model(hf_complete))                   # built-in suite
bfcl_before = run_benchmark(prompted_model(hf_complete), cases=bfcl)  # recognized suite
print('BEFORE  built-in:', before.score, '| BFCL tool acc:', bfcl_before.tool_accuracy)


## 4. Supervised fine-tune (SFT)

Each training text = the benchmark-style prompt + the gold `{"tool", "args"}` JSON
from a verified trajectory. ~5 minutes on a T4.


In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

if not hasattr(model, 'peft_config'):  # safe to re-run this cell
    model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=16, lora_dropout=0.0)

# One text per example: prompt + gold answer + EOS so the model learns to stop.
# (Unsloth's patched SFTTrainer wants a plain text column.)
def to_text(ex):
    return {'text': ex['prompt'] + '\n' + ex['completion'] + tokenizer.eos_token}

sft_ds = (load_dataset('json', data_files='tool_sft.jsonl', split='train')
          .map(to_text, remove_columns=['prompt', 'completion']))
print(sft_ds[0]['text'][-220:])

trainer = SFTTrainer(
    model=model, processing_class=tokenizer, train_dataset=sft_ds,
    args=SFTConfig(output_dir='out/sft', max_steps=150, learning_rate=2e-4,
                   per_device_train_batch_size=2, gradient_accumulation_steps=4,
                   logging_steps=25))
trainer.train()


## 5. (optional) DPO — sharpen tool choice

Preference pairs in the same format: chosen = the verified tool call, rejected =
the same args on a wrong tool. SFT alone gives the headline; flip `RUN_DPO = True`
to squeeze a bit more.


In [ ]:
RUN_DPO = False  # SFT alone gives the headline; flip to True to add DPO on top

if RUN_DPO:
    from trl import DPOConfig, DPOTrainer

    dpo_ds = load_dataset('json', data_files='dpo.jsonl', split='train')
    dpo = DPOTrainer(
        model=model, args=DPOConfig(output_dir='out/dpo', max_steps=60, learning_rate=5e-6,
                                    beta=0.1, per_device_train_batch_size=2,
                                    gradient_accumulation_steps=4, logging_steps=20),
        train_dataset=dpo_ds, processing_class=tokenizer)
    dpo.train()


## 6. Benchmark again (after) and print the before/after table


In [ ]:
after = run_benchmark(prompted_model(hf_complete))
bfcl_after = run_benchmark(prompted_model(hf_complete), cases=bfcl)

for name, b, a in [('Built-in suite', before, after),
                   ('BFCL simple_python', bfcl_before, bfcl_after)]:
    print('\n###', name)
    print(report_table_md({'before': b, 'after': a, 'n': b.n,
                           'delta_tool_accuracy': round(a.tool_accuracy - b.tool_accuracy, 4),
                           'delta_score': round(a.score - b.score, 4)}))


## 7. Publish the dataset to the Hugging Face Hub

The token must have **write** access — at https://huggingface.co/settings/tokens
create one with Token type = **Write** (a read or unscoped fine-grained token gets
`401 Unauthorized`). Run the cell, paste it when prompted. Change the repo id to
your namespace if you're not pushing to the org.


In [ ]:
from huggingface_hub import login
from agentsynth import push_dataset

login()  # paste a Hugging Face write token when prompted
url = push_dataset(result.trajectories, 'agentsynth/agentsynth-trajectories',
                   eval_results=result.eval_results)
print('dataset:', url)
